# 00 – Umgebung prüfen
Dieses Notebook prüft die technische Entwicklungsumgebung. Dokumentiert werden Python- und Bibliotheksversionen, die grundlegende Projektstruktur sowie der Git-Stand. Dadurch können spätere Ergebnisse einem definierten Software- und Codezustand zugeordnet werden.


In [1]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import json
import platform
import random
import subprocess
import sys

import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'notebooks').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Projektordner nicht gefunden.')


PROJECT_ROOT = find_project_root(Path.cwd())
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('Projekt:', PROJECT_ROOT)
print('Seed:', SEED)


Projekt: /Users/sharu/Documents/DBU/tamil-finger-sign-mlops
Seed: 42


## 1. Laufzeitumgebung

Die Pipeline verwendet versionsabhängige Bibliotheken für Feature-Extraktion, Modelltraining und Experiment Tracking. Zentralen Paketversionen werden dokumentiert. Python 3.12 und die festgelegte MediaPipe-Version werden zusätzlich geprüft, da Abweichungen die reproduzierbare Ausführung beeinflussen können.


In [2]:
PACKAGES = [
    'numpy',
    'pandas',
    'scikit-learn',
    'mediapipe',
    'mlflow',
    'Pillow',
    'joblib',
    'matplotlib',
]

EXPECTED_VERSIONS = {
    "mediapipe": "0.10.35",
    "scikit-learn": "1.9.1",
}

package_rows = []
missing = []

for package in PACKAGES:
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
        missing.append(package)
    package_rows.append({'package': package, 'version': installed})

package_versions = pd.DataFrame(package_rows)
display(package_versions)

runtime = {
    'python': platform.python_version(),
    'system': platform.system(),
    'release': platform.release(),
    'machine': platform.machine(),
}
display(pd.DataFrame([runtime]))

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f'Erwartet wird Python 3.12, gefunden wurde {platform.python_version()}.')

if missing:
    raise RuntimeError(f'Fehlende Pakete: {missing}')

mediapipe_version = package_versions.loc[
    package_versions['package'] == 'mediapipe', 'version'
].iloc[0]

if mediapipe_version != '0.10.35':
    raise RuntimeError(
        f'Erwartet wird mediapipe==0.10.35, gefunden wurde {mediapipe_version}.'
    )


,package,version
0,numpy,2.5.3
1,pandas,3.0.6
2,scikit-learn,1.9.1
3,mediapipe,0.10.35
4,mlflow,3.16.1
5,Pillow,12.3.0
6,joblib,1.6.0
7,matplotlib,3.11.2


,python,system,release,machine
0,3.12.12,Darwin,25.5.0,arm64


## 2. Projektstruktur

Vor der Ausführung wird geprüft, ob die für Daten, Artefakte, Modelle, Quellcode, Tests und Konfigurationen vorgesehenen Verzeichnisse vorhanden sind. Dadurch werden versehentliche Ausgaben an abweichende Speicherorte vermieden.

In [3]:
REQUIRED_DIRS = [
    'notebooks',
    'data/raw',
    'data/manifests',
    'artifacts',
    'models',
    'src',
    'tests',
    'configs',
]

structure = pd.DataFrame([
    {
        'path': path,
        'exists': (PROJECT_ROOT / path).is_dir(),
    }
    for path in REQUIRED_DIRS
])

display(structure)

missing_dirs = structure.loc[~structure['exists'], 'path'].tolist()
if missing_dirs:
    raise FileNotFoundError(f'Fehlende Projektordner: {missing_dirs}')


,path,exists
0,notebooks,True
1,data/raw,True
2,data/manifests,True
3,artifacts,True
4,models,True
5,src,True
6,tests,True
7,configs,True


## 3. Umgebung dokumentieren

Zusätzlich wird der aktuelle Git-Stand gespeichert, damit spätere Ergebnisse einem konkreten Code-Stand zugeordnet werden können.


In [4]:
def git_value(*args):
    result = subprocess.run(
        ['git', *args],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip() if result.returncode == 0 else None

dirty = bool(git_value("status", "--porcelain"))


git_info = {
    'commit': git_value('rev-parse', 'HEAD'),
    'branch': git_value('branch', '--show-current'),
    'dirty': dirty,
}

snapshot = {
    'project_root': str(PROJECT_ROOT),
    'seed': SEED,
    'runtime': runtime,
    'packages': dict(zip(package_versions['package'], package_versions['version'])),
    'git': git_info,
}

output_dir = PROJECT_ROOT / 'artifacts' / 'environment'
output_dir.mkdir(parents=True, exist_ok=True)

package_versions.to_csv(output_dir / '00_package_versions.csv', index=False)
(output_dir / '00_environment_snapshot.json').write_text(
    json.dumps(snapshot, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('Commit:', git_info['commit'])
print('Branch:', git_info['branch'])
print('Gespeichert:', output_dir)


Commit: ce2d6551364ba73af8b6f67c1ab171e83dae90e3
Branch: main
Gespeichert: /Users/sharu/Documents/DBU/tamil-finger-sign-mlops/artifacts/environment


## Ergebnis

Die erforderliche Projektstruktur und die für die Pipeline benötigte Softwareumgebung wurden erfolgreich geprüft. Die verwendeten Paketversionen sowie der Git-Stand werden als reproduzierbare Artefakte gespeichert.
